# 📓 Notebook 04 — CNN Model Training
## G19 Facial Emotion Recognition | CSE276

---

**Purpose of this notebook:**
- Build the CNN model defined in `src/model.py`.
- Train on the FER-2013 training split with real-time data augmentation.
- Monitor validation performance using callbacks.
- Save the best model weights to Google Drive.
- Plot training curves.

> **Prerequisite:** Notebooks 01–03 completed.
> 
> **Important:** This notebook MUST be run on **Colab with GPU enabled**.
> Go to: *Runtime → Change runtime type → T4 GPU*

In [ ]:
import sys, os
sys.path.insert(0, '/content/G19_Facial_Emotion_Recognition')

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from src.data_loader  import load_fer2013, compute_training_class_weights, EMOTION_LABELS
from src.preprocessing import apply_clahe_batch
from src.model        import build_cnn_model, get_callbacks
from src.evaluation   import plot_training_history
from src.utils        import set_random_seeds

set_random_seeds(42)

DATASET_CSV  = '/content/G19_Facial_Emotion_Recognition/dataset/fer2013.csv'
RESULTS_DIR  = '/content/G19_Facial_Emotion_Recognition/results'
MODELS_DIR   = '/content/G19_Facial_Emotion_Recognition/models'
DRIVE_MODELS = '/content/drive/MyDrive/G19_FER/models'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

# Confirm GPU
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {gpus}')
print('Setup complete.')

## 1. Load Dataset

In [ ]:
# Preserve original FER-2013 splits:
#   Training    -> training
#   PublicTest  -> validation (tuning & early stopping)
#   PrivateTest -> held out strictly for final evaluation in notebook 05
X_train, y_train, X_val, y_val, _, _ = load_fer2013(DATASET_CSV)

# Experiment toggle: baseline MUST train with USE_CLAHE = False
USE_CLAHE = False
if USE_CLAHE:
    print('Applying optional CLAHE preprocessing experiment to train & val ...')
    X_train = apply_clahe_batch(X_train)
    X_val   = apply_clahe_batch(X_val)
else:
    print('Baseline mode: training WITHOUT CLAHE.')

print(f'Train: {X_train.shape} | Val (PublicTest): {X_val.shape}')

## 2. Build Model

In [ ]:
model = build_cnn_model(
    input_shape=(48, 48, 1),
    num_classes=7,
    use_augmentation=True,
    learning_rate=1e-3,
)
model.summary()

## 3. Training Configuration

In [ ]:
# --- Hyperparameters ---
BATCH_SIZE = 64
MAX_EPOCHS = 40   # EarlyStopping will stop training early if val_loss stops improving

# --- Callbacks ---
MODEL_SAVE_PATH = f'{MODELS_DIR}/best_model.keras'
callbacks = get_callbacks(model_save_path=MODEL_SAVE_PATH)

# --- Compute class weights using ONLY training labels ---
class_weights = compute_training_class_weights(y_train)

print('Class weights (computed from training labels only):')
for idx, weight in class_weights.items():
    print(f'  {EMOTION_LABELS[idx]:<10} : {weight:.4f}')

## 4. Train the Model

In [ ]:
import time
import pandas as pd

print('Starting training ...')
print(f'Batch size : {BATCH_SIZE}')
print(f'Max epochs : {MAX_EPOCHS}')
print(f'USE_CLAHE  : {USE_CLAHE}')
print(f'Model will be saved to: {MODEL_SAVE_PATH}\n')

start_time = time.perf_counter()
history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)
training_duration_sec = time.perf_counter() - start_time

# Save per-epoch training history to CSV in results/
history_df = pd.DataFrame(history.history)
history_df.insert(0, 'epoch', range(1, len(history_df) + 1))
history_csv_path = f'{RESULTS_DIR}/baseline_training_history.csv'
history_df.to_csv(history_csv_path, index=False)

best_idx   = int(history_df['val_loss'].idxmin())
best_row   = history_df.iloc[best_idx]
final_row  = history_df.iloc[-1]

print(f'\n✅ Training complete in {training_duration_sec:.1f}s ({training_duration_sec/60:.2f} min).')
print(f'Saved training history CSV → {history_csv_path}')
print('\n--- Summary (Best Epoch by val_loss) ---')
print(f'Best epoch     : {int(best_row["epoch"])} / {len(history_df)} trained')
print(f'Train loss     : {best_row["loss"]:.4f} | Train acc : {best_row["accuracy"]*100:.2f}%')
print(f'Val loss       : {best_row["val_loss"]:.4f} | Val acc   : {best_row["val_accuracy"]*100:.2f}%')
print(f'Final epoch acc: Train={final_row["accuracy"]*100:.2f}% | Val={final_row["val_accuracy"]*100:.2f}%')

## 5. Plot Training History

In [ ]:
plot_training_history(history)

# Show inline
from PIL import Image
img = Image.open(f'{RESULTS_DIR}/training_history.png')
plt.figure(figsize=(14, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## 6. Save Model to Google Drive

In [ ]:
import shutil
os.makedirs(DRIVE_MODELS, exist_ok=True)

drive_model_path = f'{DRIVE_MODELS}/best_model.keras'
shutil.copy(MODEL_SAVE_PATH, drive_model_path)
print(f'✅ Model saved to Google Drive: {drive_model_path}')
print('\nTo use the model locally:')
print('  1. Download best_model.keras from Google Drive.')
print('  2. Place it in: G19_Facial_Emotion_Recognition/models/')
print('  3. Run the Streamlit app: streamlit run app/streamlit_app.py')

## 7. Quick Validation Accuracy Check

*(Fill in after training completes.)*

In [ ]:
# Evaluate on the validation set using the best saved model
best_model = tf.keras.models.load_model(MODEL_SAVE_PATH)

val_loss, val_acc = best_model.evaluate(X_val, y_val, verbose=0)
print(f'Best model — Val Loss    : {val_loss:.4f}')
print(f'Best model — Val Accuracy: {val_acc * 100:.2f}%')
print('\n⚠️ Full evaluation is in notebook 05_model_evaluation.ipynb')

## Summary

*(Fill in after training.)*

| Metric | Value |
|---|---|
| Epochs trained | _(fill in)_ |
| Stopped early? | _(yes/no)_ |
| Best val accuracy | _(fill in)_ |
| Best val loss | _(fill in)_ |

**Next step → Open `05_model_evaluation.ipynb`**